In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import sys
import tarfile
from pathlib import Path, PurePosixPath


def _canonical(document: object) -> bytes:
    return json.dumps(
        document,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode()


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


def _manifest(path: Path, *, schema: str) -> dict[str, object]:
    document = json.loads(path.read_bytes())
    if not isinstance(document, dict) or document.get("schema_version") != schema:
        raise RuntimeError("private input manifest schema differs")
    claimed = document.pop("manifest_sha256", None)
    if claimed != hashlib.sha256(_canonical(document)).hexdigest():
        raise RuntimeError("private input manifest digest differs")
    return document


SOURCE_INPUT = Path("/kaggle/input/apar-sentinel-v5-source3")
WHEELHOUSE_INPUT = Path("/kaggle/input/apar-sentinel-v5-wheelhouse-py312-linux-x86-64")
SAFE_INPUT = Path("/kaggle/input/apar-sentinel-v5-safe-evidence")
SOURCE_MANIFEST = _manifest(
    SOURCE_INPUT / "source-manifest.json",
    schema="apar-sentinel-v5-source-archive/1",
)
WHEELHOUSE_MANIFEST = _manifest(
    WHEELHOUSE_INPUT / "wheelhouse-manifest.json",
    schema="apar-sentinel-v5-wheelhouse/1",
)
SAFE_MANIFEST = _manifest(
    SAFE_INPUT / "safe-evidence-manifest.json",
    schema="apar-sentinel-v5-kaggle-execution-input/1",
)

SOURCE_ARCHIVE = SOURCE_INPUT / "apar-v5-source3.tar.gz"
SAFE_EVIDENCE = SAFE_INPUT / "safe-evidence.json"
if (
    SOURCE_MANIFEST.get("artifact_name") != SOURCE_ARCHIVE.name
    or SOURCE_MANIFEST.get("artifact_sha256") != _sha256(SOURCE_ARCHIVE)
):
    raise RuntimeError("source archive binding differs")
if (
    SAFE_MANIFEST.get("artifact_name") != SAFE_EVIDENCE.name
    or SAFE_MANIFEST.get("artifact_sha256") != _sha256(SAFE_EVIDENCE)
):
    raise RuntimeError("safe evidence binding differs")

wheel_entries = WHEELHOUSE_MANIFEST.get("wheels")
if not isinstance(wheel_entries, list) or not wheel_entries:
    raise RuntimeError("wheelhouse manifest is empty")
for entry in wheel_entries:
    if not isinstance(entry, dict):
        raise RuntimeError("wheelhouse entry is malformed")
    wheel = WHEELHOUSE_INPUT / str(entry.get("filename"))
    if (
        entry.get("size_bytes") != wheel.stat().st_size
        or entry.get("sha256") != _sha256(wheel)
    ):
        raise RuntimeError("wheelhouse file binding differs")

EXTRACT_ROOT = Path("/kaggle/working/apar-v5-source-extract")
if EXTRACT_ROOT.exists():
    raise RuntimeError("source extraction root already exists")
EXTRACT_ROOT.mkdir(parents=True, mode=0o700)
with tarfile.open(SOURCE_ARCHIVE, "r:gz") as archive:
    members = archive.getmembers()
    if not members:
        raise RuntimeError("source archive is empty")
    for member in members:
        relative = PurePosixPath(member.name)
        if (
            relative.is_absolute()
            or ".." in relative.parts
            or relative.parts[0] != "apar-v5-source"
        ):
            raise RuntimeError("source archive path is unsafe")
        if not (member.isfile() or member.isdir()):
            raise RuntimeError("source archive contains a non-file entry")
    archive.extractall(EXTRACT_ROOT, members=members, filter="data")
SOURCE_ROOT = EXTRACT_ROOT / "apar-v5-source"
if not (SOURCE_ROOT / "scripts/run_defense_v5_kaggle_stage.py").is_file():
    raise RuntimeError("closed stage entrypoint is absent")

NOTEBOOK_SOURCE = SOURCE_ROOT / "kaggle/defense_v5/00_authorize.ipynb"
if not NOTEBOOK_SOURCE.is_file():
    raise RuntimeError("approved notebook source is absent")
OS_RELEASE = Path("/etc/os-release")
if not OS_RELEASE.is_file():
    raise RuntimeError("Kaggle OS release binding is absent")
runtime_image_facts = {
    "schema_version": "apar-sentinel-v5-kaggle-runtime-image/1",
    "os_release_sha256": _sha256(OS_RELEASE),
    "python_executable_sha256": _sha256(Path(sys.executable)),
    "python_version": ".".join(str(item) for item in sys.version_info[:3]),
}
os.environ.update(
    {
        "APAR_V5_KAGGLE_IMAGE": "kaggle-cpu-runtime-fingerprint/1",
        "APAR_V5_KAGGLE_IMAGE_SHA256": hashlib.sha256(
            _canonical(runtime_image_facts)
        ).hexdigest(),
        "APAR_V5_DEPENDENCY_MANIFEST_SHA256": hashlib.sha256(
            (WHEELHOUSE_INPUT / "wheelhouse-manifest.json").read_bytes()
        ).hexdigest(),
        "APAR_V5_SOURCE_ARCHIVE_SHA256": _sha256(SOURCE_ARCHIVE),
        "APAR_V5_SOURCE_MANIFEST_PATH": str(
            SOURCE_INPUT / "source-manifest.json"
        ),
        "APAR_V5_NOTEBOOK_SHA256": _sha256(NOTEBOOK_SOURCE),
    }
)


In [ ]:
import subprocess
import sys

install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-input",
        "--no-index",
        "--find-links",
        str(WHEELHOUSE_INPUT),
        "--force-reinstall",
        "--no-build-isolation",
        "apar==0.1.0",
    ],
    check=False,
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    raise RuntimeError("offline dependency installation failed")


In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

CHAIN_ROOT = Path("/kaggle/working/apar-v5-chain")
CHAIN_ROOT.mkdir(mode=0o700)

OUTPUT_ROOT = CHAIN_ROOT / "00_authorize"
execution_mode = SAFE_MANIFEST.get("execution_mode")
if execution_mode not in (
    "kaggle_capacity_validation",
    "kaggle_locked_successor",
):
    raise RuntimeError("closed execution mode is absent")
command = [
    sys.executable,
    str(SOURCE_ROOT / "scripts/run_defense_v5_kaggle_stage.py"),
    "--root",
    str(SOURCE_ROOT),
    "--input-root",
    str(CHAIN_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--safe-evidence",
    str(SAFE_EVIDENCE),
    "--execution-manifest",
    str(SAFE_INPUT / "safe-evidence-manifest.json"),
    "--approved-commit",
    str(SOURCE_MANIFEST.get("approved_commit")),
]
completed = subprocess.run(
    command,
    cwd=SOURCE_ROOT,
    check=False,
    capture_output=True,
    text=True,
)
if completed.returncode != 0:
    raise RuntimeError("closed checkpoint stage failed")
receipt = json.loads(completed.stdout)
expected_receipt_keys = {
    "deterministic_sha256",
    "manifest_sha256",
    "observation_sha256",
    "stage",
}
if set(receipt) != expected_receipt_keys or receipt.get("stage") != "00_authorize":
    raise RuntimeError("redacted stage receipt differs")
print(json.dumps(receipt, sort_keys=True, separators=(",", ":")))
